# Thyroid Research Starter Analyses

This notebook contains five production-ready starter analyses using `thyroid_master.duckdb`:

1. Table 1 demographics
2. Tumor size by histology
3. FNA sensitivity/specificity
4. Tg recurrence curves (proxy)
5. Missing data heatmap

In [ ]:
from pathlib import Path
import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')

DB_PATH = Path('/Users/ros/Desktop/FInal Cleaned Thyroid Data/thyroid_master.duckdb')
con = duckdb.connect(str(DB_PATH), read_only=True)
print('Connected to:', DB_PATH)

## 1) Table 1 Demographics

In [ ]:
table1 = con.execute('''
WITH base AS (
    SELECT
        research_id,
        TRY_CAST(age_at_surgery AS DOUBLE) AS age,
        sex,
        CASE
            WHEN has_tumor_pathology THEN 'malignant'
            WHEN has_benign_pathology THEN 'benign'
            ELSE 'other'
        END AS cohort_group
    FROM master_cohort
)
SELECT
    cohort_group,
    COUNT(*) AS n_patients,
    ROUND(AVG(age), 1) AS mean_age,
    ROUND(STDDEV_POP(age), 1) AS sd_age,
    SUM(CASE WHEN LOWER(COALESCE(sex, '')) = 'female' THEN 1 ELSE 0 END) AS n_female,
    SUM(CASE WHEN LOWER(COALESCE(sex, '')) = 'male' THEN 1 ELSE 0 END) AS n_male
FROM base
GROUP BY cohort_group
ORDER BY cohort_group
''').fetchdf()

table1

## 2) Tumor Size by Histology

In [ ]:
size_hist = con.execute('''
SELECT
    histology_1_type,
    TRY_CAST(histology_1_largest_tumor_cm AS DOUBLE) AS tumor_size_cm
FROM tumor_pathology
WHERE histology_1_type IS NOT NULL
  AND TRY_CAST(histology_1_largest_tumor_cm AS DOUBLE) IS NOT NULL
''').fetchdf()

summary = (
    size_hist.groupby('histology_1_type')['tumor_size_cm']
    .agg(['count', 'mean', 'median', 'std'])
    .sort_values('count', ascending=False)
)
display(summary.head(15))

top_hist = summary.head(8).index.tolist()
plot_df = size_hist[size_hist['histology_1_type'].isin(top_hist)]

plt.figure(figsize=(12, 5))
sns.boxplot(data=plot_df, x='histology_1_type', y='tumor_size_cm')
plt.xticks(rotation=45)
plt.title('Tumor Size Distribution by Histology (Top 8)')
plt.xlabel('Histology')
plt.ylabel('Tumor Size (cm)')
plt.tight_layout()
plt.show()

## 3) FNA Sensitivity / Specificity

Definition used in this starter notebook:
- FNA positive = Bethesda 2023 >= 5
- Gold standard malignant = tumor pathology present

In [ ]:
fna_conf = con.execute('''
SELECT confusion_class, COUNT(*) AS n
FROM fna_accuracy_view
WHERE confusion_class IS NOT NULL
GROUP BY confusion_class
''').fetchdf()

counts = {r['confusion_class']: int(r['n']) for _, r in fna_conf.iterrows()}
tp = counts.get('TP', 0)
fp = counts.get('FP', 0)
fn = counts.get('FN', 0)
tn = counts.get('TN', 0)

sensitivity = tp / (tp + fn) if (tp + fn) else np.nan
specificity = tn / (tn + fp) if (tn + fp) else np.nan
ppv = tp / (tp + fp) if (tp + fp) else np.nan
npv = tn / (tn + fn) if (tn + fn) else np.nan

metrics = pd.DataFrame({
    'metric': ['sensitivity', 'specificity', 'PPV', 'NPV'],
    'value': [sensitivity, specificity, ppv, npv]
})
display(fna_conf.sort_values('confusion_class'))
display(metrics)

## 4) Tg Recurrence Curves (Proxy)

Proxy event definition (editable): first thyroglobulin value >= 2.0 ng/mL.

In [ ]:
tg = con.execute('''
SELECT
    research_id,
    days_from_first_lab,
    numeric_result
FROM longitudinal_lab_view
WHERE lab_type = 'thyroglobulin'
  AND numeric_result IS NOT NULL
  AND days_from_first_lab IS NOT NULL
ORDER BY research_id, days_from_first_lab
''').fetchdf()

event_threshold = 2.0
first_event = (
    tg[tg['numeric_result'] >= event_threshold]
    .groupby('research_id', as_index=False)['days_from_first_lab']
    .min()
    .rename(columns={'days_from_first_lab': 'event_day'})
)

all_ids = tg[['research_id']].drop_duplicates()
surv = all_ids.merge(first_event, on='research_id', how='left')

grid = np.arange(0, 3651, 90)  # ~10 years every 3 months
curve = []
n_total = len(surv)
for d in grid:
    n_event = ((surv['event_day'].notna()) & (surv['event_day'] <= d)).sum()
    curve.append({'days': d, 'event_rate': n_event / n_total if n_total else np.nan})

curve_df = pd.DataFrame(curve)

plt.figure(figsize=(10, 5))
plt.plot(curve_df['days'], curve_df['event_rate'], marker='o')
plt.title('Cumulative Tg Event Curve (Threshold >= 2.0 ng/mL)')
plt.xlabel('Days from First Tg Lab')
plt.ylabel('Cumulative Event Rate')
plt.ylim(0, 1)
plt.tight_layout()
plt.show()

curve_df.head(10)

## 5) Missing Data Heatmap

In [ ]:
missing = con.execute('''
SELECT
    has_tumor_pathology,
    has_benign_pathology,
    has_fna_cytology,
    has_ultrasound_reports,
    has_ct_imaging,
    has_mri_imaging,
    has_nuclear_med,
    has_thyroglobulin_labs,
    has_anti_thyroglobulin_labs,
    has_parathyroid
FROM master_cohort
''').fetchdf()

missing_pct = (1 - missing.mean(axis=0)) * 100
heat = pd.DataFrame(missing_pct, columns=['missing_percent']).T

plt.figure(figsize=(14, 2.8))
sns.heatmap(
    heat,
    annot=True,
    fmt='.1f',
    cmap='Reds',
    cbar_kws={'label': '% Missing'}
)
plt.title('Missingness by Data Domain (%)')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

missing_pct.sort_values(ascending=False)